# 04 Portfolio Walkforward

Notebook n?y d?ng ?? ki?m tra OOS cho **portfolio** khi b?n ?? c? m?t b? tham s? cho t?ng symbol.
N? kh?ng re-optimize trong t?ng c?a s?; m?c ti?u l? ki?m tra ?? b?n OOS c?a m?t portfolio specification ?? ch?n.


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from strategies.combo.config import summary as strategy_summary
from strategies.combo.portfolio.walkforward import walk_forward_portfolio

print(strategy_summary())


In [ ]:
SYMBOL_PARAMS = {
    'US30':   {'x': 10.0, 'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'US500':  {'x': 1.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'DE40':   {'x': 5.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'GOLD':   {'x': 0.5,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
}

ACCOUNT_MODE = 'standard'
INITIAL_BALANCE = 100_000.0
IS_BARS = 5000
OOS_BARS = 1250
STEP_BARS = 1250
MAX_BARS = 40000


In [ ]:
wf_df, wf_summary = walk_forward_portfolio(
    SYMBOL_PARAMS,
    account_mode=ACCOUNT_MODE,
    initial_balance=INITIAL_BALANCE,
    is_bars=IS_BARS,
    oos_bars=OOS_BARS,
    step_bars=STEP_BARS,
    max_bars=MAX_BARS,
)

print(wf_summary)
display(wf_df)


In [ ]:
if not wf_df.empty:
    metric_cols = [c for c in ['window','oos_start','oos_end','total_return','max_drawdown','sharpe','profit_factor','win_rate'] if c in wf_df.columns]
    display(wf_df[metric_cols])

    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)
    if 'total_return' in wf_df.columns:
        wf_df.plot(x='window', y='total_return', kind='bar', ax=axes[0], color='#6BCB77', legend=False, title='OOS total return by window')
        axes[0].grid(alpha=0.3)
    if 'max_drawdown' in wf_df.columns:
        wf_df.plot(x='window', y='max_drawdown', kind='bar', ax=axes[1], color='#FF6B6B', legend=False, title='OOS max drawdown by window')
        axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Walk-forward returned no rows. Increase max_bars or reduce window sizes.')
